In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

# 1. Define the CNN Architecture
class MNISTCNN(nn.Module):
    def __init__(self):
        super(MNISTCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2) 
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        
        # Two poolings turn 28x28 -> 14x14 -> 7x7
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.dropout1(x)
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

# 2. Setup Hyperparameters & Data Loaders
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
epochs = 10
learning_rate = 0.001

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 3. Initialize Model, Loss, and Optimizer
model = MNISTCNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.NLLLoss()

# 4. Training and Evaluation Loop
print(f"Starting training on device: {device}\n")

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    train_correct = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", unit="batch")
    
    for data, target in progress_bar:
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # Track training accuracy metrics
        pred = output.argmax(dim=1, keepdim=True)
        train_correct += pred.eq(target.view_as(pred)).sum().item()
        
        progress_bar.set_postfix(loss=f"{running_loss / len(progress_bar):.4f}")
        
    # Calculate overall metrics for the training set
    train_accuracy = 100. * train_correct / len(train_loader.dataset)
        
    # --- VALIDATION STEP ---
    model.eval()
    test_loss = 0
    val_correct = 0
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            val_correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader)
    val_accuracy = 100. * val_correct / len(test_loader.dataset)
    
    # Print the clean summary dashboard for this epoch
    print(f"[Epoch {epoch} Metrics] -> "
          f"Train Acc: {train_accuracy:.2f}% | "
          f"Val Acc: {val_accuracy:.2f}% | "
          f"Val Loss: {test_loss:.4f}\n")

print("Training finished successfully!")

# 5. Save the Weights for your Web App
torch.save(model.state_dict(), "mnist_cnn.pth")
print("\nModel saved as mnist_cnn.pth")

Starting training on device: cuda



Epoch 1/10: 100%|██████████| 938/938 [00:16<00:00, 56.84batch/s, loss=0.2246]


[Epoch 1 Metrics] -> Train Acc: 93.09% | Val Acc: 98.57% | Val Loss: 0.0452



Epoch 2/10: 100%|██████████| 938/938 [00:16<00:00, 57.62batch/s, loss=0.0884]


[Epoch 2 Metrics] -> Train Acc: 97.43% | Val Acc: 98.59% | Val Loss: 0.0396



...

[Epoch 8 Metrics] -> Train Acc: 98.89% | Val Acc: 99.17% | Val Loss: 0.0235



Epoch 9/10: 100%|██████████| 938/938 [00:16<00:00, 57.87batch/s, loss=0.0340]


[Epoch 9 Metrics] -> Train Acc: 99.00% | Val Acc: 99.32% | Val Loss: 0.0211



Epoch 10/10: 100%|██████████| 938/938 [00:16<00:00, 56.91batch/s, loss=0.0323]


[Epoch 10 Metrics] -> Train Acc: 98.99% | Val Acc: 99.33% | Val Loss: 0.0206

Training finished successfully!

Model saved as mnist_cnn.pth
